# Tutorial: A06 Losses, LAE, Development, Trend, and Adjustments

Audience:
- CAS Exam 5 pricing students using a code-first workflow.

Prerequisites:
- Basic familiarity with pandas and core ratemaking terminology.
- Access to the local `chainladder-python` repository in this project.

Learning goals:
- Work with paid, case, reported/incurred, and closed claim perspectives.
- Demonstrate development and trend timing to average accident/earned date.
- Outline large-loss, cap/limit, catastrophe, and claim-count adjustments.


## Outline

1. Setup and data loading
2. Formula-sheet alignment
3. Exhibit builder scaffold
4. Selection narrative prompts
5. EXAM RED FLAG checks
6. Mini drill

Notebook standard alignment:
- Keep the rhythm: markdown intent -> code exhibit -> markdown interpretation.
- Show intermediate steps; do not hide core calculations.
- Apply display conventions: currency `{:,.0f}`, factors `{:,.4f}`, percentages `{:.1%}`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd


def find_repo_path(name: str) -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidate = base / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate folder: {name}")


chainladder_repo = find_repo_path("chainladder-python")
if str(chainladder_repo) not in sys.path:
    sys.path.insert(0, str(chainladder_repo))

try:
    import chainladder as cl
    CHAINLADDER_AVAILABLE = True
except ModuleNotFoundError:
    cl = None
    CHAINLADDER_AVAILABLE = False

SAMPLE_DATA_DIR = chainladder_repo / "chainladder" / "utils" / "data"


def load_sample(name: str):
    if CHAINLADDER_AVAILABLE:
        return cl.load_sample(name)
    csv_path = SAMPLE_DATA_DIR / f"{name}.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Sample dataset not found: {csv_path}")
    return pd.read_csv(csv_path)


def to_frame(obj) -> pd.DataFrame:
    if hasattr(obj, "to_frame"):
        return obj.to_frame().reset_index()
    return pd.DataFrame(obj).copy()


def numeric_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]


DISPLAY_FORMATS = {
    "currency": "{:,.0f}",
    "factor": "{:,.4f}",
    "percent": "{:.1%}",
}

pd.options.display.max_columns = 30
print(f"chainladder import available: {CHAINLADDER_AVAILABLE}")


## Formula-sheet alignment (Werner Ch 6 - Losses and LAE)

Relevant formulas for this notebook:
- `Reported (Incurred) Loss = Paid Loss + Case Reserve`
- `Ultimate Loss = Reported Loss x Selected CDF`
- `Trended Ultimate = Ultimate Loss x Loss Trend Factor`
- `Adjusted Loss = Trended Loss after large-loss/cat/benefit adjustments`

Terms to define in your final solution:
- Numerator and denominator choices.
- Time period alignment (CY/AY/PY/report-year as applicable).
- Which assumptions are selected vs. directly observed.


## Chainladder dataset plan

Primary sample data for this chapter:
- `berqsherm`, `xyz`, `quarterly`

Use this notebook to connect Werner chapter mechanics to reproducible sample-data exhibits.


In [ ]:
DATASETS = ["berqsherm", "xyz", "quarterly"]
triangles = {name: load_sample(name) for name in DATASETS}

for name, tri in triangles.items():
    frame = to_frame(tri)
    print(f"{name}: rows={frame.shape[0]}, cols={frame.shape[1]}")
    print(f"  first columns: {list(frame.columns[:8])}")


## Exhibit builder

Create a single function that reproduces the canonical table for this chapter.
Replace the generic grouping and fields with the exact exhibit structure you want to practice for exam-style prompts.


In [ ]:
def build_exhibit(
    df: pd.DataFrame,
    group_cols: list[str] | None = None,
    value_cols: list[str] | None = None,
) -> pd.DataFrame:
    # Scaffold for chapter exhibit construction.
    group_cols = [col for col in (group_cols or []) if col in df.columns]
    value_cols = [col for col in (value_cols or numeric_columns(df)[:4]) if col in df.columns]

    if group_cols and value_cols:
        return (
            df.groupby(group_cols, dropna=False)[value_cols]
            .sum()
            .reset_index()
            .sort_values(group_cols)
        )

    if value_cols:
        summary = df[value_cols].describe().T
        return summary.reset_index().rename(columns={"index": "metric"})

    return df.head(10).copy()


primary_name = DATASETS[0]
primary_df = to_frame(triangles[primary_name])
exhibit_preview = build_exhibit(primary_df)
exhibit_preview.head(10)


## Selection narrative

Document your choices before finalizing indications:
- Key selection to justify: Which development/trend selections are supportable for the valuation maturity.
- What data evidence supports the selection?
- What alternative selection would you test and why?
- Show sensitivity: how does indicated change move if this assumption shifts by +/- 5%?


## EXAM RED FLAG - checklist

- Trending from valuation date instead of average accident/earned date.
- Applying both development and case adequacy adjustments on the same issue.
- Removing large losses without a stated replacement method.


## Mini drill

1. Conceptual: When is explicit development unnecessary in pricing?
1. Computation: Convert paid to reported and then to selected ultimate.
1. What-if: Recompute indication with one catastrophe year excluded.
